# Batch Experiment Manager — dataset-aware

Use this notebook after executing the Stage 0 data-processing notebook once. It injects workload/run metadata into each model notebook and consolidates metrics.

In [ ]:
# ============================================================
# Batch Experiment Manager - model evaluation with scientific tracking
# ============================================================
# Stage 0 is the data-processing notebook. It is executed once outside this runner.
# This runner only references the Stage 0 output through DATASET_ID and DATASET_MANIFEST_PATH.

import os
import re
import gc
import json
import time
import shutil
import platform
import traceback
from pathlib import Path
from datetime import datetime

import nbformat
import pandas as pd
from nbconvert.preprocessors import ExecutePreprocessor

try:
    import torch
except Exception:
    torch = None

# ============================================================
# 1) Configuration
# ============================================================

# Project root according to the current CNSM_2026 organization.
PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", r"C:\CNSM_2026"))
NOTEBOOKS_DIR = Path(globals().get("NOTEBOOKS_DIR", PROJECT_ROOT / "notebooks"))
RESULTS_ROOT = Path(globals().get("RESULTS_ROOT", PROJECT_ROOT / "results"))
SHARED_ARTIFACTS_ROOT = Path(globals().get("SHARED_ARTIFACTS_ROOT", PROJECT_ROOT / "shared_artifacts"))
DADO_PICKLE_ROOT = Path(globals().get("DADO_PICKLE_ROOT", PROJECT_ROOT / "dado_pickle"))

notebooks_dir = NOTEBOOKS_DIR
results_root = RESULTS_ROOT

# Stage 0: dataset processing output. Fill/update these after running the processing notebook once.
DATASET_ID = str(globals().get("DATASET_ID", "manual_stage0_dataset"))
DATASET_MANIFEST_PATH = Path(globals().get(
    "DATASET_MANIFEST_PATH",
    RESULTS_ROOT / "datasets" / DATASET_ID / "dataset_manifest.json"
))
PROCESSING_RUN_DIR = Path(globals().get("PROCESSING_RUN_DIR", DATASET_MANIFEST_PATH.parent))
PROCESSED_DATA_ROOT = Path(globals().get("PROCESSED_DATA_ROOT", RESULTS_ROOT / "datasets"))

notebooks = [
 #   "1BLSTM.ipynb",
 #   "2Dense.ipynb",
  #  "3CNN_LSTM.ipynb",
  #  "4BGRU.ipynb",
  #  "5GNN.ipynb",
    "6GCN_GRU.ipynb",
    "7CNN.ipynb",
]

datasets = [
    "cjpeg-rose7-preset.exe",
    "core.exe",
    "linear_alg-mid-100x100-sp.exe",
    "loops-all-mid-10k-sp.exe",
    "nnet_test.exe",
    "parser-125k.exe",
    "radix2-big-64k.exe",
    "sha-test.exe",
    "zip-test.exe",
]

KERNEL_NAME = "python3"
TIMEOUT_SECONDS = None
IOPUB_TIMEOUT_SECONDS = 3600
ALLOW_ERRORS = False
CONTINUE_ON_ERROR = True
CLEAR_GPU_BETWEEN_RUNS = True

# ============================================================
# 2) Helpers
# ============================================================

def now_id() -> str:
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def safe_name(value: str) -> str:
    value = str(value).replace(".exe", "")
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", value)
    return value.strip("_") or "unknown"


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_json(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, default=str)


def append_csv_row(path: Path, row: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    exists = path.exists()
    pd.DataFrame([row]).to_csv(path, mode="a", header=not exists, index=False)


def replace_software_in_cell_source(source: str, dataset: str) -> str:
    pattern = r"(?m)^(\s*software\s*=\s*)(['\"])(.*?)(\2)\s*$"
    replacement = rf"\1\2{dataset}\2"
    new_source, count = re.subn(pattern, replacement, source, count=1)
    return new_source if count else source


def inject_execution_context(nb, *, notebook_name: str, software: str, batch_id: str,
                             execution_number: int, run_id: str, results_root: Path, run_dir: Path):
    context_lines = [
        "# Auto-injected by Batch Experiment Manager.",
        "from pathlib import Path",
        f"NOTEBOOK_NAME = {notebook_name!r}",
        f"software = {software!r}",
        f"BATCH_ID = {batch_id!r}",
        f"EXECUTION_NUMBER = {execution_number!r}",
        f"RUN_ID = {run_id!r}",
        f"PROJECT_ROOT = Path(r'{str(PROJECT_ROOT)}')",
        f"NOTEBOOKS_DIR = Path(r'{str(NOTEBOOKS_DIR)}')",
        f"RESULTS_ROOT = Path(r'{str(results_root)}')",
        f"SHARED_ARTIFACTS_ROOT = Path(r'{str(SHARED_ARTIFACTS_ROOT)}')",
        f"DADO_PICKLE_ROOT = Path(r'{str(DADO_PICKLE_ROOT)}')",
        f"RUN_DIR = Path(r'{str(run_dir)}')",
        f"DATASET_ID = {DATASET_ID!r}",
        f"DATASET_MANIFEST_PATH = Path(r'{str(DATASET_MANIFEST_PATH)}')",
        f"PROCESSING_RUN_DIR = Path(r'{str(PROCESSING_RUN_DIR)}')",
        f"PROCESSED_DATA_ROOT = Path(r'{str(PROCESSED_DATA_ROOT)}')",
    ]
    nb.cells.insert(0, nbformat.v4.new_code_cell("\n".join(context_lines)))

    replaced = False
    for cell in nb.cells[1:]:
        if cell.cell_type == "code" and re.search(r"(?m)^\s*software\s*=", cell.source):
            new_source = replace_software_in_cell_source(cell.source, software)
            if new_source != cell.source:
                cell.source = new_source
                replaced = True
                break
    return replaced


def extract_notebook_text_outputs(nb) -> str:
    chunks = []
    for i, cell in enumerate(nb.cells):
        if cell.cell_type != "code":
            continue
        for output in cell.get("outputs", []):
            out_type = output.get("output_type")
            if out_type == "stream":
                text = output.get("text", "")
                if text:
                    chunks.append(f"\n--- cell {i} stream:{output.get('name', '')} ---\n{text}")
            elif out_type in {"execute_result", "display_data"}:
                data = output.get("data", {})
                text = data.get("text/plain")
                if text:
                    if isinstance(text, list):
                        text = "".join(text)
                    chunks.append(f"\n--- cell {i} {out_type} ---\n{text}")
            elif out_type == "error":
                traceback_text = "\n".join(output.get("traceback", []))
                chunks.append(f"\n--- cell {i} error ---\n{traceback_text}")
    return "\n".join(chunks)


def read_run_summary(run_dir: Path) -> pd.DataFrame:
    candidates = [run_dir / "test_metrics_summary.csv", *run_dir.glob("**/test_metrics_summary.csv")]
    seen = set()
    for p in candidates:
        if p in seen:
            continue
        seen.add(p)
        if p.exists():
            try:
                return pd.read_csv(p)
            except Exception:
                pass
    return pd.DataFrame()


def clear_gpu_memory():
    if torch is None:
        return
    try:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


def get_environment_payload():
    payload = {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "project_root": str(PROJECT_ROOT),
        "notebooks_dir": str(notebooks_dir),
        "results_root": str(results_root),
        "shared_artifacts_root": str(SHARED_ARTIFACTS_ROOT),
        "dado_pickle_root": str(DADO_PICKLE_ROOT),
        "kernel_name": KERNEL_NAME,
        "dataset_id": DATASET_ID,
        "dataset_manifest_path": str(DATASET_MANIFEST_PATH),
        "processing_run_dir": str(PROCESSING_RUN_DIR),
    }
    if torch is not None:
        payload.update({
            "torch_version": getattr(torch, "__version__", None),
            "cuda_available": torch.cuda.is_available(),
            "cuda_version": getattr(torch.version, "cuda", None),
            "cudnn_version": torch.backends.cudnn.version() if torch.cuda.is_available() else None,
            "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        })
    return payload

# ============================================================
# 3) Create batch folders
# ============================================================

batch_id = f"batch_{now_id()}"
batch_root = ensure_dir(results_root / "batches" / batch_id)
executed_dir = ensure_dir(batch_root / "executed_notebooks")
logs_dir = ensure_dir(batch_root / "logs")
errors_dir = ensure_dir(batch_root / "errors")
runs_dir = ensure_dir(batch_root / "runs")

if not DATASET_MANIFEST_PATH.exists():
    # Create a lightweight manifest from the already processed shared_artifacts folders.
    # This avoids blocking the model batch when Stage 0 did not write a manifest file.
    DATASET_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    software_dirs = sorted([p.name for p in SHARED_ARTIFACTS_ROOT.iterdir() if p.is_dir()]) if SHARED_ARTIFACTS_ROOT.exists() else []
    fallback_manifest = {
        "dataset_id": DATASET_ID,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "created_by": "batch_manager_fallback_manifest",
        "note": "Stage 0 manifest was missing; this manifest records the processed shared_artifacts currently available.",
        "project_root": str(PROJECT_ROOT),
        "shared_artifacts_root": str(SHARED_ARTIFACTS_ROOT),
        "software_dirs": software_dirs,
        "expected_files_per_software": ["X_hex_y_pad256.npz", "indices_pad256_seed42.npz"],
    }
    save_json(DATASET_MANIFEST_PATH, fallback_manifest)
    print(f"Created fallback DATASET_MANIFEST_PATH: {DATASET_MANIFEST_PATH}")

batch_config = {
    "batch_id": batch_id,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "notebooks": notebooks,
    "datasets": datasets,
    "project_root": str(PROJECT_ROOT),
    "notebooks_dir": str(notebooks_dir),
    "results_root": str(results_root),
    "shared_artifacts_root": str(SHARED_ARTIFACTS_ROOT),
    "dado_pickle_root": str(DADO_PICKLE_ROOT),
    "batch_root": str(batch_root),
    "dataset_id": DATASET_ID,
    "dataset_manifest_path": str(DATASET_MANIFEST_PATH),
    "processing_run_dir": str(PROCESSING_RUN_DIR),
    "processed_data_root": str(PROCESSED_DATA_ROOT),
    "kernel_name": KERNEL_NAME,
    "timeout_seconds": TIMEOUT_SECONDS,
    "iopub_timeout_seconds": IOPUB_TIMEOUT_SECONDS,
    "allow_errors": ALLOW_ERRORS,
    "continue_on_error": CONTINUE_ON_ERROR,
}
save_json(batch_root / "batch_config.json", batch_config)
save_json(batch_root / "batch_environment.json", get_environment_payload())

if DATASET_MANIFEST_PATH.exists():
    shutil.copy2(DATASET_MANIFEST_PATH, batch_root / "dataset_manifest.snapshot.json")

manifest_rows = []
execution_number = 0
for nb_name in notebooks:
    for dataset in datasets:
        execution_number += 1
        run_id = f"{batch_id}_exec{execution_number:03d}_{safe_name(Path(nb_name).stem)}_{safe_name(dataset)}"
        manifest_rows.append({
            "batch_id": batch_id,
            "execution_number": execution_number,
            "run_id": run_id,
            "dataset_id": DATASET_ID,
            "notebook_name": nb_name,
            "software": dataset,
            "planned_status": "planned",
        })
manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(batch_root / "execution_manifest.csv", index=False)

print(f"Batch ID: {batch_id}")
print(f"Batch directory: {batch_root}")
print(f"Dataset ID: {DATASET_ID}")
print(f"Planned executions: {len(manifest_df)}")
manifest_df.head()

# ============================================================
# 4) Execute notebooks
# ============================================================

status_path = batch_root / "execution_status.csv"
batch_summary_path = batch_root / "batch_results_summary.csv"
global_summary_path = results_root / "results_summary_model.csv"

executor = ExecutePreprocessor(
    timeout=TIMEOUT_SECONDS,
    kernel_name=KERNEL_NAME,
    allow_errors=ALLOW_ERRORS,
    iopub_timeout=IOPUB_TIMEOUT_SECONDS,
)

total = len(manifest_rows)
for row in manifest_rows:
    nb_name = row["notebook_name"]
    dataset = row["software"]
    execution_number = int(row["execution_number"])
    run_id = row["run_id"]
    notebook_path = notebooks_dir / nb_name
    run_dir = runs_dir / f"exec{execution_number:03d}_{safe_name(Path(nb_name).stem)}_{safe_name(dataset)}"
    ensure_dir(run_dir)

    executed_notebook_path = executed_dir / f"{run_id}_executed.ipynb"
    error_notebook_path = errors_dir / f"{run_id}_PARTIAL_ERROR.ipynb"
    log_path = logs_dir / f"{run_id}.txt"

    start_time = datetime.now()
    t0 = time.perf_counter()
    status = "started"
    error_type = ""
    error_message = ""

    print(f"\n[{execution_number:03d}/{total:03d}] {nb_name} | {dataset}")

    try:
        if not notebook_path.exists():
            raise FileNotFoundError(f"Notebook not found: {notebook_path}")

        nb = nbformat.read(notebook_path, as_version=4)
        software_line_replaced = inject_execution_context(
            nb,
            notebook_name=nb_name,
            software=dataset,
            batch_id=batch_id,
            execution_number=execution_number,
            run_id=run_id,
            results_root=batch_root,
            run_dir=run_dir,
        )

        nb.metadata.setdefault("batch_execution", {})
        nb.metadata["batch_execution"].update({
            "batch_id": batch_id,
            "execution_number": execution_number,
            "run_id": run_id,
            "dataset_id": DATASET_ID,
            "software": dataset,
            "software_line_replaced": software_line_replaced,
            "start_time": start_time.isoformat(timespec="seconds"),
        })

        executor.preprocess(nb, {"metadata": {"path": str(PROJECT_ROOT)}})

        status = "success"
        end_time = datetime.now()
        elapsed_sec = time.perf_counter() - t0
        nb.metadata["batch_execution"].update({
            "status": status,
            "end_time": end_time.isoformat(timespec="seconds"),
            "elapsed_sec": elapsed_sec,
        })
        nbformat.write(nb, executed_notebook_path)
        log_path.write_text(extract_notebook_text_outputs(nb), encoding="utf-8")

        run_summary = read_run_summary(run_dir)
        if not run_summary.empty:
            run_summary.insert(0, "batch_root", str(batch_root))
            run_summary.insert(0, "executed_notebook_path", str(executed_notebook_path))
            run_summary.insert(0, "log_path", str(log_path))
            run_summary.to_csv(batch_summary_path, mode="a", header=not batch_summary_path.exists(), index=False)
            run_summary.to_csv(global_summary_path, mode="a", header=not global_summary_path.exists(), index=False)
        else:
            print(f"Warning: test_metrics_summary.csv not found for {run_id}: {run_dir}")

    except Exception as exc:
        status = "failed"
        error_type = exc.__class__.__name__
        error_message = str(exc)
        try:
            if "nb" in locals():
                nb.metadata.setdefault("batch_execution", {})
                nb.metadata["batch_execution"].update({
                    "status": status,
                    "error_type": error_type,
                    "error_message": error_message,
                    "elapsed_sec": time.perf_counter() - t0,
                })
                nbformat.write(nb, error_notebook_path)
                partial_log = extract_notebook_text_outputs(nb)
            else:
                partial_log = ""
        except Exception:
            partial_log = ""
        log_path.write_text(partial_log + "\n\n--- Python exception ---\n" + traceback.format_exc(), encoding="utf-8")
        print(f"Failed: {nb_name} | {dataset}")
        print(f"{error_type}: {error_message}")
        if not CONTINUE_ON_ERROR:
            raise

    finally:
        status_row = {
            "batch_id": batch_id,
            "execution_number": execution_number,
            "run_id": run_id,
            "dataset_id": DATASET_ID,
            "notebook_name": nb_name,
            "software": dataset,
            "status": status,
            "start_time": start_time.isoformat(timespec="seconds"),
            "end_time": datetime.now().isoformat(timespec="seconds"),
            "elapsed_sec": time.perf_counter() - t0,
            "error_type": error_type,
            "error_message": error_message,
            "run_dir": str(run_dir),
            "executed_notebook_path": str(executed_notebook_path if status == "success" else error_notebook_path),
            "log_path": str(log_path),
        }
        append_csv_row(status_path, status_row)
        if CLEAR_GPU_BETWEEN_RUNS:
            clear_gpu_memory()

print("Batch execution completed")
print(f"Status table: {status_path}")
print(f"Batch summary: {batch_summary_path}")
print(f"Global summary: {global_summary_path}")

parou no 6 e 7 para todos os benchmarks. dense para zip deu erro.